# Model Evaluation Notebook

This notebook evaluates the baseline DR ATE estimator on simulated data and reports metrics for $\hat\tau$: bias, RMSE, MSE, average SE, and coverage.

In [ ]:
from __future__ import annotations

import csv
import math
from pathlib import Path

from dGC.gen_data import sample_data
from dGC.baseline import estimate_tau_hat_dr
from dGC.ate import tau_hat_from_gnn
from dGC.utils import doubly_robust_scores, mean, mse, standard_error


In [ ]:
def evaluate_baseline_dr(
    num_replications: int,
    sample_size: int,
    graph_model: str = "rgg",
    seed: int = 123,
    true_tau: float = 0.0,
    feature_key: str = "X",
) -> dict:
    tau_hats = []
    se_hats = []
    cover_flags = []

    for r in range(num_replications):
        draw = sample_data(
            sample_size=sample_size,
            seed=seed + r,
            graph_model=graph_model,
        )
        fit = estimate_tau_hat_dr(draw, feature_key=feature_key)

        tau_hat = fit["tau_hat"]
        psi = doubly_robust_scores(
            draw["Y"],
            draw.get("D", draw["T"]),
            fit["mu1_hat"],
            fit["mu0_hat"],
            fit["p_hat"],
        )
        se_hat = standard_error(psi)

        tau_hats.append(tau_hat)
        se_hats.append(se_hat)
        cover_flags.append(abs(tau_hat - true_tau) <= 1.96 * se_hat)

    mean_tau = mean(tau_hats)
    mse_tau = mse(tau_hats, true_tau)

    return {
        "num_replications": int(num_replications),
        "sample_size": int(sample_size),
        "graph_model": graph_model,
        "true_tau": float(true_tau),
        "mean_tau_hat": mean_tau,
        "bias": mean_tau - true_tau,
        "mse_tau_hat": mse_tau,
        "rmse_tau_hat": math.sqrt(mse_tau),
        "avg_se_hat": mean(se_hats),
        "coverage_95": mean([1.0 if c else 0.0 for c in cover_flags]),
    }


def evaluate_from_gnn_outputs(data: dict, gnn_outputs: dict, true_tau: float = 0.0) -> dict:
    """Evaluate one set of GNN nuisance outputs."""
    out = tau_hat_from_gnn(gnn_outputs, data, return_details=True)
    tau_hat = out["tau_hat"]
    return {
        "tau_hat": tau_hat,
        "sq_error": (tau_hat - true_tau) ** 2,
        "se": out["se"],
        "estimator": out["estimator"],
        "n": out["n"],
    }


In [ ]:
NUM_REPLICATIONS = 50
SAMPLE_SIZES = [100, 300, 500]
GRAPH_MODELS = ["rgg", "er"]
SEED = 123
TRUE_TAU = 0.0
FEATURE_KEY = "X"


In [ ]:
results = []
for gm in GRAPH_MODELS:
    for n in SAMPLE_SIZES:
        row = evaluate_baseline_dr(
            num_replications=NUM_REPLICATIONS,
            sample_size=n,
            graph_model=gm,
            seed=SEED,
            true_tau=TRUE_TAU,
            feature_key=FEATURE_KEY,
        )
        results.append(row)

results


In [ ]:
out_path = Path("results/model_evaluation.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "num_replications",
    "sample_size",
    "graph_model",
    "true_tau",
    "mean_tau_hat",
    "bias",
    "mse_tau_hat",
    "rmse_tau_hat",
    "avg_se_hat",
    "coverage_95",
]

with out_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print(f"Saved evaluation table to {out_path.resolve()}")


## Optional: Evaluate a single GNN output

If you have `mu1`, `mu0`, and optionally propensity `p` from your GNN, run:

```python
draw = sample_data(sample_size=500, seed=123, graph_model="rgg")
gnn_outputs = {
    "mu1": [...],
    "mu0": [...],
    # "p": [...]  # optional
}
evaluate_from_gnn_outputs(draw, gnn_outputs, true_tau=0.0)
```